# 🔍 Expresiones Regulares — Validación y Búsqueda
## Algoritmos y Estructuras de Datos I — UADE

Las expresiones regulares (**regex**) son patrones de texto que permiten buscar y validar cadenas de caracteres de forma precisa.

En este notebook las aplicamos a dos casos concretos:

| Caso de uso | Función re | Descripción |
|---|---|---|
| ✅ Validación de datos | `re.fullmatch()` | Verificar que nombre, email e id tengan el formato correcto |
| 🔎 Búsqueda de temas | `re.search()` | Encontrar temas por nombre o autor a partir de un fragmento |

> 💡 La librería `re` viene incluida en Python, no hace falta instalar nada.

In [ ]:
import re

## ¿Por qué usamos `r""` en los patrones?

- Python interpreta `\` como inicio de secuencia de escape: `\n` es salto de línea, `\t` es tab.
- En regex, `.` significa "cualquier carácter" y `\.` significa "punto literal". Si escribimos `"\."` sin raw string, Python primero procesa `\.` → `\` + `.` pero puede generar comportamientos inesperados con caracteres como `\b`, `\d`, `\w`.
- Con `r"patron"` Python **NO** procesa los `\`, los pasa tal cual al motor de regex.

| Escrito en código | Sin `r""` Python interpreta | Con `r""` regex recibe |
|---|---|---|
| `"\."` | `\.` → punto literal ✅ (funciona igual) | `\.` → punto literal ✅ |
| `"\d"` | `\d` → no es secuencia conocida, se pasa igual ⚠️ | `\d` → dígito ✅ |
| `"\b"` | `\b` → **backspace** ❌ | `\b` → límite de palabra ✅ |
| `"\w"` | `\w` → no es secuencia conocida ⚠️ | `\w` → letra/dígito/_ ✅ |

**Regla simple: siempre usá `r""` para patrones regex.**

## Elementos básicos de regex

### Tabla 1 — Metacaracteres principales

| Elemento | Significado | Ejemplo de patrón | Coincide con |
|---|---|---|---|
| `.` | cualquier carácter (excepto `\n`) | `a.c` | `"abc"`, `"a3c"` |
| `\.` | punto literal | `re\.fullmatch` | `"re.fullmatch"` |
| `^` | inicio de cadena | `^Hola` | `"Hola mundo"` |
| `$` | fin de cadena | `mundo$` | `"Hola mundo"` |
| `\d` | dígito `[0-9]` | `\d\d` | `"42"` |
| `\w` | letra, dígito o `_` | `\w+` | `"abc"`, `"a1_"` |
| `\b` | límite de palabra | `\bBTS\b` | `"BTS"` pero no `"ABTS"` |

### Tabla 2 — Cuantificadores

| Cuantificador | Significado | Ejemplo |
|---|---|---|
| `+` | uno o más | `[a-z]+` → `"abc"` |
| `*` | cero o más | `[0-9]*` → `""`, `"123"` |
| `?` | cero o uno | `[A-Z]?` → `""`, `"A"` |
| `{n}` | exactamente n | `[0-9]{4}` → `"1234"` |
| `{n,m}` | entre n y m | `[0-9]{2,4}` → `"12"` … `"1234"` |

### Tabla 3 — Conjuntos y grupos

| Elemento | Significado | Ejemplo |
|---|---|---|
| `[a-z]` | cualquier letra minúscula | `"a"`, `"z"` |
| `[A-Z]` | cualquier letra mayúscula | `"A"`, `"Z"` |
| `[A-Za-z]` | cualquier letra | cubre mayúsculas y minúsculas |
| `[0-9]` o `\d` | cualquier dígito | `"0"` … `"9"` |
| `[abc]` | a, b o c | `[aeiou]` → vocal |
| `[^abc]` | cualquier carácter EXCEPTO a, b, c | `[^0-9]` → no dígito |
| `(ab\|cd)` | grupo: ab o cd | `(jpg\|png)` → `"jpg"` o `"png"` |


---
## Bloque 1 — `re.fullmatch()` para validación

`re.fullmatch(patron, cadena)` retorna un **Match object** si *toda* la cadena coincide con el patrón, o `None` si no coincide.

```python
re.fullmatch(patron, cadena)         # valida la cadena completa
bool(re.fullmatch(patron, cadena))   # True / False
```

Ideal para validar datos de entrada: nombre, email, id, teléfono.

### 1.1 Validar nombre

Patrón: `r"[A-Za-záéíóúÁÉÍÓÚüÜñÑ ]+"`

- `[A-Za-z...]` — conjunto de letras: mayúsculas, minúsculas, acentuadas (á, é, í, ó, ú), ñ, ü
- el **espacio** dentro del conjunto permite nombres compuestos como `"Ana García"` o `"José María"`
- `+` — al menos un carácter (descarta la cadena vacía)

Este patrón rechaza dígitos, signos de puntuación y caracteres especiales.

In [ ]:
patron_nombre = r"[A-Za-záéíóúÁÉÍÓÚüÜñÑ ]+"

def validar_nombre(nombre: str) -> bool:
    return bool(re.fullmatch(patron_nombre, nombre.strip()))

# Probamos casos válidos e inválidos
casos = ["Ana García", "José María", "ana123", "Pedro!", ""]
for c in casos:
    resultado = "✅ válido" if validar_nombre(c) else "❌ inválido"
    print(f"  '{c}' → {resultado}")

  'Ana García' → ✅ válido
  'José María' → ✅ válido
  'ana123' → ❌ inválido
  'Pedro!' → ❌ inválido
  '' → ❌ inválido


### 1.2 Validar email

Construimos el patrón por partes:

| Parte | Patrón | Descripción |
|---|---|---|
| usuario | `[\w.+-]+` | letras, dígitos, punto, guión, + |
| @ | `@` | literal |
| dominio + extensión | `[\w-]+(\.​[A-Za-z0-9-]+)+` | permite `gmail.com` y `uade.edu.ar` |

El fragmento `(\.​[A-Za-z0-9-]+)+` permite **múltiples segmentos** separados por punto → acepta subdominios como `uade.edu.ar`.

Patrón completo: `r"[\w.+-]+@[\w-]+(\.​[A-Za-z0-9-]+)+"`

In [ ]:
patron_email = r"[\w.+-]+@[\w-]+(\.[A-Za-z0-9-]+)+"

def validar_email(email: str) -> bool:
    return bool(re.fullmatch(patron_email, email.strip()))

casos = [
    "ana@gmail.com",
    "maria@uade.edu.ar",     # subdominio → ✅ válido
    "carlos@yahoo.com",
    "sin_arroba.com",
    "doble@@dominio.com",
    "falta_extension@dominio",
]
for c in casos:
    resultado = "✅ válido" if validar_email(c) else "❌ inválido"
    print(f"  '{c}' → {resultado}")

  'ana@gmail.com' → ✅ válido
  'maria@uade.edu.ar' → ✅ válido
  'carlos@yahoo.com' → ✅ válido
  'sin_arroba.com' → ❌ inválido
  'doble@@dominio.com' → ❌ inválido
  'falta_extension@dominio' → ❌ inválido


### 1.3 Validar id_tema

Contexto: cuando el usuario ingresa el id con `input()`, el valor llega como **string**. Antes de convertirlo con `int()`, hay que verificar que sea un número entero positivo.

Patrón: `r"[1-9][0-9]*"`

- `[1-9]` — primer dígito entre 1 y 9 (descarta el `0` como id válido)
- `[0-9]*` — cero o más dígitos adicionales (permite `1`, `42`, `100`, etc.)

Este patrón rechaza `0`, negativos, decimales y cualquier texto.

In [ ]:
patron_id = r"[1-9][0-9]*"

def validar_id_tema(id_tema: str) -> bool:
    return bool(re.fullmatch(patron_id, id_tema.strip()))

# Casos
casos = ["1", "42", "0", "-3", "abc", "3.5", ""]
for c in casos:
    resultado = "✅ válido" if validar_id_tema(c) else "❌ inválido"
    print(f"  '{c}' → {resultado}")

  '1' → ✅ válido
  '42' → ✅ válido
  '0' → ❌ inválido
  '-3' → ❌ inválido
  'abc' → ❌ inválido
  '3.5' → ❌ inválido
  '' → ❌ inválido


---
## Bloque 2 — `re.search()` para búsquedas parciales

`re.search(patron, cadena)` busca el patrón en **cualquier parte** de la cadena. Retorna un Match object si lo encuentra, o `None` si no.

```python
re.search(patron, cadena)                    # case-sensitive
re.search(patron, cadena, re.IGNORECASE)     # ignora mayúsculas/minúsculas
```

Ideal para búsquedas donde el usuario escribe solo un fragmento.

In [ ]:
temas = [
    {"id_tema": 1, "nombre": "Dai Dai",       "autor": "Shakira"},
    {"id_tema": 2, "nombre": "Love Yourself",  "autor": "BTS"},
    {"id_tema": 3, "nombre": "Waka Waka",      "autor": "Shakira"},
    {"id_tema": 4, "nombre": "Fake Love",      "autor": "BTS"},
    {"id_tema": 5, "nombre": "Positions",      "autor": "Ariana Grande"},
]
print(f"Dataset cargado: {len(temas)} temas")

Dataset cargado: 5 temas


### 2.1 Búsqueda por nombre (fragmento)

El texto ingresado por el usuario se usa **directamente como patrón** en `re.search`. Con `re.IGNORECASE` la búsqueda ignora mayúsculas y minúsculas: `"dai"` encuentra `"Dai Dai"`.

In [ ]:
def buscar_tema_por_nombre(nombre: str, temas: list) -> list:
    return [t for t in temas if re.search(nombre, t["nombre"], re.IGNORECASE)]

# Evaluamos los mismos casos del Playground
for busqueda in ["Dai Dai", "Dai", "dai", "da", "Love"]:
    resultados = buscar_tema_por_nombre(busqueda, temas)
    nombres = [t["nombre"] for t in resultados]
    print(f"  '{busqueda}' → {nombres}")

  'Dai Dai' → ['Dai Dai']
  'Dai' → ['Dai Dai']
  'dai' → ['Dai Dai']
  'da' → ['Dai Dai', 'Waka Waka']
  'Love' → ['Love Yourself', 'Fake Love']


### 2.2 Búsqueda con patrón: inicio de palabra con `^` y `\b`

| Patrón | Qué significa | Encuentra |
|---|---|---|
| `^Love` | la cadena **entera** empieza con `"Love"` | `"Love Yourself"` ✅ — `"Fake Love"` ❌ |
| `Love$` | la cadena **entera** termina con `"Love"` | `"Fake Love"` ✅ — `"Love Yourself"` ❌ |
| `\bLove` | `"Love"` aparece al inicio de **cualquier palabra** | `"Love Yourself"` ✅ y `"Fake Love"` ✅ |

El **word boundary** `\b` marca el límite entre un carácter de palabra (`\w`) y uno que no lo es (espacio, inicio, fin de cadena). Por eso `\bLove` encuentra `"Love"` tanto al inicio como después de un espacio.

In [ ]:
# Comienza con Love (inicio de cadena)
patron_inicio = r"^Love"
resultados = [t for t in temas if re.search(patron_inicio, t["nombre"], re.IGNORECASE)]
print(f"Comienza con 'Love': {[t['nombre'] for t in resultados]}")

# Termina con Love (fin de cadena)
patron_fin = r"Love$"
resultados = [t for t in temas if re.search(patron_fin, t["nombre"], re.IGNORECASE)]
print(f"Termina con 'Love': {[t['nombre'] for t in resultados]}")

# Love como inicio de palabra (word boundary)
patron_wb = r"\bLove"
resultados = [t for t in temas if re.search(patron_wb, t["nombre"], re.IGNORECASE)]
print(f"Palabra que empieza con 'Love': {[t['nombre'] for t in resultados]}")

Comienza con 'Love': ['Love Yourself']
Termina con 'Love': ['Fake Love']
Palabra que empieza con 'Love': ['Love Yourself', 'Fake Love']


### 2.3 Búsqueda por autor

Misma lógica que la búsqueda por nombre, pero aplicada al campo `"autor"`. Con `re.IGNORECASE` el usuario puede escribir `"ari"` y encontrar `"Ariana Grande"` sin importar mayúsculas.

In [ ]:
def buscar_tema_por_autor(autor: str, temas: list) -> list:
    return [t for t in temas if re.search(autor, t["autor"], re.IGNORECASE)]

for busqueda in ["Shakira", "BTS", "ari"]:
    resultados = buscar_tema_por_autor(busqueda, temas)
    print(f"  '{busqueda}' → {[(t['nombre'], t['autor']) for t in resultados]}")

  'Shakira' → [('Dai Dai', 'Shakira'), ('Waka Waka', 'Shakira')]
  'BTS' → [('Love Yourself', 'BTS'), ('Fake Love', 'BTS')]
  'ari' → [('Positions', 'Ariana Grande')]
